# EnvPool

> Steps several multi-agent envs together and stacks everything into one info dict.

In [ ]:
#| default_exp world.env_pool

In [ ]:
#| hide
from nbdev.showdoc import *

The multi-agent counterpart of stable-worldmodel's `EnvPool`. Like its `EverythingToInfoWrapper`,
the pool puts everything a policy or a dataset needs into one **info dict** per step:

| key | shape | content |
|---|---|---|
| observation keys (`image`, `pov`, `direction`, ...) | `(num_envs, 1, num_agents, ...)` | each agent's observation |
| `reward`, `terminated`, `truncated` | `(num_envs, 1, num_agents)` | of the step that led here (`nan`, False after a reset) |
| `action` | `(num_envs, 1, num_agents)` | the action that led here (`nan` after a reset) |
| `state` | `(num_envs, 1, ...)` | global state (`env.state()`) |
| `pixels` | `(num_envs, 1, H, W, 3)` | render of the whole env (if `add_pixels`) |
| `step_idx`, `id`, `seed` | `(num_envs, 1)` | step in the episode, episode id, reset seed (-1: none) |

The info buffers are allocated on the first reset and written in place afterwards (copy what
you keep). Envs keep their per-agent dict API; only the pool deals with stacking.

In [ ]:
#| export
from __future__ import annotations

from typing import Any, Callable, Sequence

import gymnasium as gym
import numpy as np
from gymnasium import spaces
from gymnasium.vector.utils import batch_space

from stable_marl.envs.base import MultiAgentEnv

In [ ]:
#| export
PER_AGENT_STEP_KEYS = ('reward', 'terminated', 'truncated', 'action')


def joint_action_space(env: MultiAgentEnv) -> spaces.Space:
    "Action space of all the agents of `env` as one space: MultiDiscrete if they are all Discrete."
    agent_spaces = [env.action_space[a] for a in range(env.num_agents)]
    if all(isinstance(s, spaces.Discrete) for s in agent_spaces):
        return spaces.MultiDiscrete([s.n for s in agent_spaces], start=[s.start for s in agent_spaces])
    return spaces.Tuple(agent_spaces)


class EnvPool:
    """
    Runs ``num_envs`` multi-agent envs side by side, one after the other in this process.

    :meth:`reset` and :meth:`step` take a `mask` to only reset / step some envs: the others keep
    their info and report zero reward and no termination.

    Parameters
    ----------
    env_fns : list of callables
        One factory per env, each called once
    add_pixels : bool
        Add the env's render (``env.render()``, needs ``render_mode='rgb_array'``) as ``pixels``
    """
    def __init__(self, env_fns: Sequence[Callable[[], MultiAgentEnv]], add_pixels: bool = False):
        self.envs: list[MultiAgentEnv] = [fn() for fn in env_fns]
        assert self.envs, "EnvPool needs at least one env"
        self.num_agents: int = self.envs[0].unwrapped.num_agents
        assert all(env.unwrapped.num_agents == self.num_agents for env in self.envs), \
            "All envs of a pool must have the same number of agents"
        self.add_pixels = add_pixels
        self.seeds = np.full(self.num_envs, -1, dtype=np.int64)
        self._single_action_space = joint_action_space(self.envs[0].unwrapped)
        # cached: a batched space rebuilt on each access would never advance its RNG
        self._action_space = batch_space(self._single_action_space, self.num_envs)
        self._infos: dict[str, Any] | None = None
        self._ids, self._step_idx = np.zeros(self.num_envs, np.int64), np.zeros(self.num_envs, np.int64)

    @property
    def num_envs(self) -> int:
        return len(self.envs)

    @property
    def action_space(self) -> spaces.Space:
        "Batched action space: samples have shape (num_envs, num_agents)."
        return self._action_space

    @property
    def single_action_space(self) -> spaces.Space:
        "Joint action space of the agents of one env."
        return self._single_action_space

    @property
    def single_observation_space(self) -> spaces.Dict:
        "Observation space of one env (agent id -> agent observation space)."
        return self.envs[0].observation_space

    @property
    def state_space(self) -> spaces.Space:
        "Global state space of one env."
        return self.envs[0].unwrapped.state_space

    @property
    def variation_space(self):
        "Variation space of the envs, or None if they do not define one."
        return getattr(self.envs[0].unwrapped, 'variation_space', None)

    def close(self):
        for env in self.envs:
            env.close()

In [ ]:
#| export
def _broadcast_arg(arg, n: int, increment: bool = False) -> list:
    "One value per env: lists are used as is, an int seed becomes `seed + i`, anything else is shared."
    if isinstance(arg, (list, tuple, np.ndarray)):
        assert len(arg) == n, f"Expected {n} values, got {len(arg)}"
        return list(arg)
    if increment and isinstance(arg, (int, np.integer)):
        return [int(arg) + i for i in range(n)]
    return [arg] * n


def _is_array_like(value) -> bool:
    return isinstance(value, (np.ndarray, np.generic, bool, int, float)) and not isinstance(value, str)


def _stack_agents(values: list) -> Any:
    "Stack the values of the agents of one env (a list if they are not arrays)."
    return np.stack([np.asarray(v) for v in values]) if all(_is_array_like(v) for v in values) else list(values)


def episode_done(terminateds: np.ndarray, truncateds: np.ndarray) -> np.ndarray:
    "Whether each env's episode is over (every agent terminated or truncated), shape (num_envs,)."
    return (np.asarray(terminateds) | np.asarray(truncateds)).all(axis=1)

In [ ]:
#| export
from fastcore.utils import patch

@patch
def _env_info(self: EnvPool, i: int, obs: dict, rewards, terminations, truncations, actions, infos: dict) -> dict:
    "Everything about env `i` at this step, unstacked (see the table above)."
    env, A = self.envs[i], self.num_agents
    info = {key: _stack_agents([obs[a][key] for a in range(A)]) for key in obs[0]}
    for key in (infos.get(0) or {}):    # the env's own per-agent infos
        info.setdefault(key, _stack_agents([infos[a].get(key) for a in range(A)]))
    info['reward'] = np.array([rewards[a] for a in range(A)], np.float32)
    info['terminated'] = np.array([terminations[a] for a in range(A)], bool)
    info['truncated'] = np.array([truncations[a] for a in range(A)], bool)
    info['action'] = np.asarray(actions, np.float32).reshape(A)
    info['state'] = env.unwrapped.state()
    if self.add_pixels:
        info['pixels'] = env.render()
    info['step_idx'], info['id'], info['seed'] = self._step_idx[i], self._ids[i], self.seeds[i]
    return info

@patch
def _write(self: EnvPool, i: int, info: dict):
    "Write the info of env `i` into the stacked buffers (allocated on first use)."
    if self._infos is None:
        self._infos = {key: (np.zeros((self.num_envs, 1, *np.shape(v)), np.asarray(v).dtype)
                             if _is_array_like(v) or isinstance(v, np.ndarray)
                             else [[None] for _ in range(self.num_envs)])
                       for key, v in info.items()}
    for key, value in info.items():
        self._infos[key][i][0] = value

@patch
def reset(
    self: EnvPool,
    seed: int | Sequence[int | None] | None = None,
    options: dict | Sequence[dict | None] | None = None,
    mask: np.ndarray | None = None) -> tuple[None, dict]:
    """
    Reset the envs (only those where `mask` is True, if given) and return ``(None, infos)``.

    `seed` is an int (env `i` gets ``seed + i``), one seed per env, or None.
    `options` is one dict for every env, or one per env.
    """
    if self._infos is None and mask is not None and not np.all(mask):
        raise RuntimeError("The first reset of an EnvPool must reset every env")
    seeds, options = _broadcast_arg(seed, self.num_envs, increment=True), _broadcast_arg(options, self.num_envs)
    nan = [np.nan] * self.num_agents
    for i in (range(self.num_envs) if mask is None else np.flatnonzero(mask)):
        obs, infos = self.envs[i].reset(seed=seeds[i], options=options[i])
        self.seeds[i] = -1 if seeds[i] is None else seeds[i]
        rng = np.random.default_rng(None if seeds[i] is None else [seeds[i], 0x5EED])
        self._ids[i], self._step_idx[i] = rng.integers(np.iinfo(np.int64).max), 0
        false = [False] * self.num_agents
        self._write(i, self._env_info(i, obs, nan, false, false, nan, infos))
    return None, self._infos

@patch
def step(
    self: EnvPool,
    actions: np.ndarray,
    mask: np.ndarray | None = None) -> tuple[None, np.ndarray, np.ndarray, np.ndarray, dict]:
    """
    Step the envs (only those where `mask` is True, if given) with `actions` of shape
    ``(num_envs, num_agents)``. Returns ``(None, rewards, terminateds, truncateds, infos)``,
    the first three of shape ``(num_envs, num_agents)``.
    """
    actions = np.asarray(actions)
    assert actions.shape[:2] == (self.num_envs, self.num_agents), \
        f"Expected actions of shape ({self.num_envs}, {self.num_agents}), got {actions.shape}"
    shape = (self.num_envs, self.num_agents)
    rewards, terminateds, truncateds = np.zeros(shape, np.float32), np.zeros(shape, bool), np.zeros(shape, bool)
    for i in (range(self.num_envs) if mask is None else np.flatnonzero(mask)):
        act = {a: actions[i, a].item() if actions[i, a].ndim == 0 else actions[i, a] for a in range(self.num_agents)}
        obs, rew, term, trunc, infos = self.envs[i].step(act)
        self._step_idx[i] += 1
        info = self._env_info(i, obs, rew, term, trunc, actions[i], infos)
        self._write(i, info)
        rewards[i], terminateds[i], truncateds[i] = info['reward'], info['terminated'], info['truncated']
    return None, rewards, terminateds, truncateds, self._infos

### Tests

In [ ]:
from functools import partial
from stable_marl.envs import make

pool = EnvPool([partial(make, 'MultiGrid-Empty-6x6-v0', agents=2, render_mode='rgb_array')] * 3, add_pixels=True)
_, infos = pool.reset(seed=10)
assert list(pool.seeds) == [10, 11, 12] and list(infos['seed'][:, 0]) == [10, 11, 12]
assert infos['image'].shape == (3, 1, 2, 7, 7, 3) and infos['pov'].shape == (3, 1, 2, 224, 224, 3)
assert infos['state'].shape == (3, 1, 6, 6, 3) and infos['pixels'].shape == (3, 1, 192, 192, 3)
assert np.isnan(infos['reward']).all() and np.isnan(infos['action']).all() and not infos['terminated'].any()
assert pool.action_space.sample().shape == (3, 2) and pool.single_action_space.shape == (2,)

# the pool matches stepping the envs one by one
solo = [make('MultiGrid-Empty-6x6-v0', agents=2) for _ in range(3)]
for i, env in enumerate(solo): env.reset(seed=10 + i)
rng = np.random.default_rng(0)
for t in range(20):
    actions = rng.integers(3, size=(3, 2))
    _, rew, term, trunc, infos = pool.step(actions)
    for i, env in enumerate(solo):
        o, r, te, tr, _ = env.step({a: int(actions[i, a]) for a in range(2)})
        for a in range(2):
            assert np.array_equal(infos['pov'][i, 0, a], o[a]['pov']) and rew[i, a] == r[a] and term[i, a] == te[a]
        assert np.array_equal(infos['state'][i, 0], env.state()) and infos['step_idx'][i, 0] == t + 1
    assert np.array_equal(infos['action'][:, 0], actions) and np.array_equal(infos['reward'][:, 0], rew)

# masked envs are not stepped
before = infos['image'].copy()
_, rew, term, trunc, infos = pool.step(np.full((3, 2), 2), mask=np.array([True, False, False]))
assert np.array_equal(infos['image'][1:], before[1:]) and not rew[1:].any() and not term[1:].any()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()